# Linear Regression & Optimization (CSC 422)

**Today:** we have some points. We want the line. The interesting part is not the
line — it is *how a computer finds it without being told the answer.*

**Duration:** 50 minutes · **Format:** live coding, run every cell together

| | | |
|---|---|---|
| **0–6** | The problem | points on a screen |
| **6–14** | What makes a line *good*? | the loss |
| **14–24** | Brute force | try every line, and watch it not scale |
| **24–34** | Follow the slope | the gradient, then 8 lines of descent |
| **34–44** | Watch it learn | the path down the bowl |
| **44–50** | One knob ruins everything | the learning rate |

**Instructor note:** every code cell below is short on purpose. Run it, look at
the picture, say the one sentence, move on. The payoff cell is the 8-line
gradient descent loop at 24 min — everything before it exists to make those 8
lines feel inevitable.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(422)
plt.rcParams["figure.figsize"] = (7, 4.5)

---
## 0–6 min · The problem

Robby tracked how many cups of coffee he drank against how many problems he
got through. Here is the data. **Where is the line?**

In [ ]:
# 40 students, coffee vs problems solved
x = np.random.uniform(0, 4, 40)
y = 2.5 * x - 1.0 + np.random.normal(0, 1.2, 40)     # the truth, plus noise

plt.scatter(x, y, s=45, alpha=.75, edgecolor="k", linewidth=.5)
plt.xlabel("cups of coffee"); plt.ylabel("problems solved")
plt.title("Where is the line?"); plt.grid(alpha=.2); plt.show()

**Ask the class:** *"Everyone point at the slope. What is it, roughly?"*

They will say something between 2 and 3. They are doing regression in their
heads. The rest of today is teaching a computer to do the same thing — and
the computer does not get to eyeball it.

---
## 6–14 min · What makes a line *good*?

A model is just a rule with knobs. Ours has two: slope `a`, intercept `b`.

In [ ]:
def predict(a, b, x):
    return a * x + b

def loss(a, b, x, y):
    return np.mean((y - predict(a, b, x)) ** 2)      # mean squared error

Two candidate lines. **Which one is better — and by how much?**

In [ ]:
print(f"Line A   y = 2.0x + 0.0    loss = {loss(2.0, 0.0, x, y):.3f}")
print(f"Line B   y = 3.0x - 2.0    loss = {loss(3.0, -2.0, x, y):.3f}")

In [ ]:
for a_, b_, c in [(2.0, 0.0, "crimson"), (3.0, -2.0, "teal")]:
    plt.plot(x, predict(a_, b_, x), color=c, lw=2,
             label=f"y={a_}x+{b_}   loss={loss(a_,b_,x,y):.2f}")
plt.scatter(x, y, s=40, alpha=.6, edgecolor="k", linewidth=.5, zorder=3)
plt.legend(); plt.grid(alpha=.2); plt.title("Two guesses, two numbers"); plt.show()

**The point.** "Which line looks better" is an opinion. **Loss turns it into a
number**, and a number is something a computer can act on.

Everything from here is the same question: *make that number small.*

---
## 14–24 min · Brute force: just try every line

Obvious idea. Two knobs, so lay a grid over both and score every combination.

In [ ]:
a_grid = np.linspace(-1, 5, 120)
b_grid = np.linspace(-4, 2, 120)
A, B = np.meshgrid(a_grid, b_grid)

# score all 120x120 lines at once
L = np.mean((y[:, None, None] - (A * x[:, None, None] + B)) ** 2, axis=0)

i, j = np.unravel_index(L.argmin(), L.shape)
print(f"{L.size:,} lines tried")
print(f"best:  a = {A[i,j]:.2f},  b = {B[i,j]:.2f},  loss = {L[i,j]:.3f}")
print(f"truth: a = 2.50,  b = -1.00")

That worked. Now **look at the shape of what we just searched** — this picture
is the whole course.

In [ ]:
from matplotlib.colors import LogNorm

plt.contourf(A, B, L, levels=np.logspace(np.log10(L.min()), np.log10(L.max()), 40),
             norm=LogNorm(), cmap="viridis")
plt.colorbar(label="loss (log scale)")
plt.contour(A, B, L, levels=np.logspace(np.log10(L.min()), np.log10(L.max()), 14),
            colors="white", linewidths=.4, alpha=.5)
plt.plot(A[i,j], B[i,j], "r*", ms=20, label="best line found")
plt.xlabel("slope a"); plt.ylabel("intercept b")
plt.title("The loss surface — a long, narrow valley"); plt.legend(); plt.show()

**The point.** The loss surface is a **bowl**. There is a bottom, and the bottom
is the answer. We did not have to be clever to find it — we just checked
everywhere.

So why not always do this?

In [ ]:
for knobs, name in [(2, "our line"), (100, "a small model"), (1_000_000, "a real network")]:
    print(f"{name:>16s}: {knobs:>9,} knobs  ->  10^{knobs:,} lines to check")

**The point.** Grid search dies instantly. Ten values per knob and two knobs is
100 lines. A million knobs is $10^{1000000}$ — more combinations than atoms in
the universe, by an absurd margin.

We cannot check everywhere. **We need to start somewhere and walk downhill.**

---
## 24–34 min · Follow the slope

Standing on the side of a bowl in the dark, you do not need a map. You only
need to know **which way is down**, and the slope under your feet tells you.

The slope of the loss with respect to each knob:

$$\frac{\partial L}{\partial a} = \frac{2}{n}\sum x_i\,(\hat y_i - y_i)
\qquad
\frac{\partial L}{\partial b} = \frac{2}{n}\sum (\hat y_i - y_i)$$

In [ ]:
def gradients(a, b, x, y):
    err = predict(a, b, x) - y
    return 2 * np.mean(x * err), 2 * np.mean(err)

**Check it does what we think.** Take a slice through the bowl at `b = -1` and
ask the gradient which way is downhill from three different places.

In [ ]:
aa = np.linspace(-1, 5, 200)
plt.plot(aa, [loss(a_, -1.0, x, y) for a_ in aa], color="0.6", lw=2)

for a_ in [0.0, 2.5, 4.5]:
    g, _ = gradients(a_, -1.0, x, y)
    plt.arrow(a_, loss(a_, -1, x, y), -g * 0.06, 0,          # length tracks |gradient|
              head_width=1.6, head_length=.12, color="crimson", lw=2,
              length_includes_head=True)
    plt.plot(a_, loss(a_, -1, x, y), "ko", ms=7)
    plt.annotate(f"slope {g:+.1f}", (a_, loss(a_, -1, x, y)),
                 textcoords="offset points", xytext=(0, 12), ha="center", fontsize=9)

plt.xlabel("slope a"); plt.ylabel("loss"); plt.grid(alpha=.2)
plt.title("Arrow = $-$gradient. Long where it is steep, tiny at the bottom."); plt.show()

**The point.** The gradient points *uphill*. Negate it and you are pointed at
the answer — from anywhere on the curve, without ever seeing the whole picture.

That is the entire idea. Here it is in code.

In [ ]:
def fit(x, y, lr=0.1, steps=300):
    a, b = 0.0, 0.0                       # start anywhere
    path = [(a, b)]
    for _ in range(steps):
        ga, gb = gradients(a, b, x, y)    # which way is up?
        a -= lr * ga                      # step the other way
        b -= lr * gb
        path.append((a, b))
    return a, b, np.array(path)

a_hat, b_hat, path = fit(x, y)

print(f"grid search        a={A[i,j]:.4f}  b={B[i,j]:.4f}   loss={L[i,j]:.6f}   {L.size:,} lines")
print(f"gradient descent   a={a_hat:.4f}  b={b_hat:.4f}   loss={loss(a_hat,b_hat,x,y):.6f}      300 steps")

**The point.** That is **eight lines**, and it beat a 14,400-point grid search
using 300 steps — *and it found a better line*, because the grid can only ever
land on one of its own gridpoints.

It also does not care whether there are 2 knobs or 2 billion; the loop is
identical. This is how every model in this course is trained, including the
transformer in week 13.

**Worth pausing on.** We generated this data from `a = 2.5, b = -1.0`, but the
best possible line for the 40 points we actually drew is `a = 2.62, b = -1.38`.

Gradient descent did not get it wrong — it found the genuine minimum. The
*sample* is not the *truth*, and no optimiser can recover a signal the noise
has hidden. Fitting the data you have as well as possible is not the same as
recovering reality. That gap is the whole of Module 05.

---
## 34–44 min · Watch it learn

Two views of the same run. First: the path it walked down the bowl.

In [ ]:
plt.contourf(A, B, L, levels=np.logspace(np.log10(L.min()), np.log10(L.max()), 40),
             norm=LogNorm(), cmap="viridis", alpha=.9)
plt.contour(A, B, L, levels=np.logspace(np.log10(L.min()), np.log10(L.max()), 14),
            colors="white", linewidths=.4, alpha=.45)
plt.plot(path[:,0], path[:,1], "w.-", lw=1.6, ms=3, label="the walk")
plt.plot(0, 0, "wo", ms=11, mec="k", label="start")
plt.plot(a_hat, b_hat, "r*", ms=20, label="finish")
plt.xlabel("slope a"); plt.ylabel("intercept b")
plt.title("It rolled to the bottom"); plt.legend(loc="lower left"); plt.show()

Second: what that walk looked like as an actual line on the data.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3.4), sharey=True)
for ax, step in zip(axes, [0, 3, 10, 100]):
    a_, b_ = path[step]
    ax.scatter(x, y, s=22, alpha=.5)
    ax.plot(x, predict(a_, b_, x), "crimson", lw=2.5)
    ax.set_title(f"step {step}   loss {loss(a_,b_,x,y):.2f}")
    ax.grid(alpha=.2)
plt.tight_layout(); plt.show()

**Ask the class:** *"Between step 3 and step 10, what changed more — the slope
or the intercept?"*

Look back at the contour plot. The bowl is **steeper along `a` than along `b`**,
so the slope gets fixed first and the intercept drifts into place afterwards.
Gradient descent fixes whatever is most wrong first. Nobody told it to.

---
## 44–50 min · One knob ruins everything

`lr` is how big a step we take. It is the only thing we have not questioned.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.6))
for ax, lr in zip(axes, [0.005, 0.1, 0.55]):
    _, _, p = fit(x, y, lr=lr, steps=60)
    losses = [loss(a_, b_, x, y) for a_, b_ in p]
    ax.plot(losses, lw=2, color="crimson")
    ax.set_title(f"lr = {lr}"); ax.set_xlabel("step"); ax.grid(alpha=.2)
    ax.set_yscale("log")
axes[0].set_ylabel("loss (log scale)")
plt.tight_layout(); plt.show()

**The point.**

| | |
|---|---|
| `lr = 0.005` | correct, and far too slow to be useful |
| `lr = 0.1` | drops like a stone |
| `lr = 0.55` | **the loss goes up** — it steps over the bottom and climbs the far wall |

Same data, same code, same starting point. One number.

This is not a toy failure: in **CA.01** a learning rate of 0.1 trains perfectly
well on scaled features and sends the parameters to $10^{273}$ on raw ones. And
**PS1 problem 3** asks you to run exactly this by hand and find the threshold
where it flips.

---
## Where this goes

You now know how every model in this course is trained:

1. **A model** with knobs — today, a line with two
2. **A loss** that scores how wrong it is — today, mean squared error
3. **The gradient**, which says which way is downhill
4. **A step** in the opposite direction, repeated

A neural network changes exactly one of these: the model gets bigger. The loss,
the gradient and the step are the same four lines you wrote today. Backprop
(Module 03) is nothing but the chain rule for computing step 3 when the model
has millions of knobs instead of two.

**Due Wednesday:** PS1 — problem 3 is this lecture, by hand, on two numbers.